## Import Libraries


In [ ]:
# from langchain_unstructured import UnstructuredLoader
# from langchain_ollama import OllamaEmbeddings, ChatOllama
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_chroma import Chroma

# from langchain_core.prompts import   ChatPromptTemplate, PromptTemplate
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.runnables import RunnablePassthrough
# from langchain_classic.retrievers import MultiQueryRetriever

In [2]:
!pip install langchain_ollama


   ---------------------------------------- 0/2 [ollama]
   -------------------- ------------------- 1/2 [langchain_ollama]
   -------------------- ------------------- 1/2 [langchain_ollama]
   ---------------------------------------- 2/2 [langchain_ollama]




[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
!pip uninstall unstructured-inference

In [ ]:
# Imports
from langchain_community.document_loaders import UnstructuredPDFLoader, PyPDFLoader, PyMuPDFLoader
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import   ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.retrievers import MultiQueryRetriever
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Jupyter-specific imports
from IPython.display import display, Markdown

# Set environment variable for protobuf
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

## Load PDF

In [3]:
# Load PDF
local_path = r"D:\All_Project\Sem_Project\DocuMind_Private_RAG\data\pdf\encrption_numerical.pdf"

if local_path:
    loader = PyPDFLoader(file_path=local_path)
    data = loader.load()
    print(f"PDF loaded successfully: {local_path}")
else:
    print("Upload a PDF file")

PDF loaded successfully: D:\All_Project\Sem_Project\DocuMind_Private_RAG\data\pdf\encrption_numerical.pdf


## Split text into chunks

In [4]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(data)
print(f"Text split into {len(chunks)} chunks")

Text split into 2 chunks


## embedding model

In [5]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)
# test embedding
vector = embeddings.embed_query("What is AI?")

print(len(vector))
print(vector[:5])

768
[-0.00029012564, 0.07100799, -0.14482684, -0.01660038, 0.07352834]


## create vector database

In [6]:
# Create vector database
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="local-rag"
)
print("Vector database created successfully")

Vector database created successfully


## set LLM

In [7]:
# Set up LLM and retrieval
local_model = "llama3.2:3b" 
llm = ChatOllama(model=local_model)


## set prompt template and retriever

In [8]:
# Query prompt template
QUERY_PROMPT = PromptTemplate(
    template="""You are an AI language model assistant. Your task is to generate 2
    different versions of the given user question to retrieve relevant documents from
    a vector database. By generating multiple perspectives on the user question, your
    goal is to help the user overcome some of the limitations of the distance-based
    similarity search. Provide these alternative questions separated by newlines.
    Original question: {question}""",
    input_variables=["question"],
)

# Set up retriever
retriever = MultiQueryRetriever.from_llm(
    vector_db.as_retriever(), 
    llm,
    prompt=QUERY_PROMPT
)

In [9]:
# RAG prompt template
template = """Answer the question based ONLY on the following context: {context} Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

## create chain

In [10]:
# Create chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## chat with PDF

In [11]:
def chat_with_pdf(question):
    """
    Chat with the PDF using the RAG chain.
    """
    return display(Markdown(chain.invoke(question)))

In [12]:
# question 1.
chat_with_pdf("What is the main idea of this document?")

The main idea of this document appears to be a study guide or textbook on cryptography, specifically covering various types of encryption techniques such as substitution ciphers, transposition ciphers, stream ciphers, and block ciphers. The document is divided into sections, each focusing on a different aspect of cryptography, and includes questions and problems for the reader to work on.

In [13]:
# question 2
chat_with_pdf("Can you list main topic in the document?")

The main topics in the document are:

1. Substitution Ciphers
2. Transposition Ciphers
3. Stream Ciphers
4. Block Ciphers

## clean db

In [ ]:
# Clean up when done 
vector_db.delete_collection()
print("Vector database deleted successfully")